# Практическое занятие 13. Проведение частотного анализа слов, а также анализа биграмм и триграмм

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/p13.ipynb)

**Цели занятия:**

- Строить частотные словари с помощью `collections.Counter` и частотные таблицы в pandas (абсолютная и относительная частота)
- Извлекать биграммы и триграммы (через `zip` и `nltk.util.ngrams`) без пересечения границ предложений
- Фильтровать n-граммы со стоп-словами и сравнивать частотные профили двух категорий текстов

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/OWNER/REPO.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Частотный анализ

**Частотный анализ** — первый шаг исследования любого корпуса: какие слова встречаются чаще всего, какие — редко, чем отличаются тексты разных тем. На частотах построены ключевые слова, тематические модели, языковые модели и поисковые системы.

```python
from collections import Counter
tokens = ["talabalar", "va", "oʻqituvchilar", "va", "talabalar"]
freq = Counter(tokens)
freq["talabalar"]          # 2
freq["kitob"]              # 0 — для отсутствующего слова не KeyError, а 0
freq.most_common(2)        # [('talabalar', 2), ('va', 2)]
freq.update(["kitob"])     # дописать новые токены
```

> **Совет: Нормализуйте и запрос**  
> Слова в частотном словаре записаны канонически (*oʻzbek*). Искать `freq["o'zbek"]` бесполезно — получите 0. Пропускайте запрос через ту же нормализацию: `freq[normalize("o'zbek")]`.

### Абсолютная и относительная частота

| Мера | Формула | Когда нужна |
|---|---|---|
| абсолютная частота | `count` | внутри одного корпуса |
| относительная частота | `count / N` | сравнение корпусов разного размера |
| на 1000 слов / **ipm** | `count / N * 1000` (или `* 1 000 000`) | удобные «человеческие» числа |

Сравнивать *абсолютные* частоты двух категорий некорректно: в большей категории все слова будут «чаще». Поэтому частоты нормируют на размер категории.

```python
freq_df = pd.DataFrame(freq.most_common(), columns=["word", "count"])
freq_df["rel_freq"] = freq_df["count"] / freq_df["count"].sum()
freq_df.head(15).plot.barh(x="word", y="count")
```

## N-граммы

**N-грамма** — последовательность из *n* соседних токенов: униграммы (1), **биграммы** (2), **триграммы** (3). Они показывают устойчивые сочетания: *sunʼiy intellekt*, *oʻzbek tili*, *katta maʼlumotlar*.

Из `N` токенов получается `N − n + 1` n-грамм. В Python их удобно строить функцией `zip` со сдвигом:

```python
toks = ["oʻzbek", "tili", "uchun", "til", "modellari"]
bigrams  = list(zip(toks, toks[1:]))              # [('oʻzbek','tili'), ('tili','uchun'), ...]
trigrams = list(zip(toks, toks[1:], toks[2:]))

from nltk.util import ngrams                      # то же из библиотеки NLTK
list(ngrams(toks, 3))                             # без загрузки корпусов NLTK
```

> **Внимание: Границы предложений**  
> Если склеить все токены текста в один список, появятся ложные биграммы из конца одного предложения и начала следующего: *«... ochildi. Ochilish ...»* → `('ochildi', 'ochilish')`. Правильно — сначала делить текст на предложения и строить n-граммы **внутри** каждого.

```python
def sentence_ngrams(text, n):
    result = []
    for sent in split_sentences(text):
        t = tokenize(sent)
        result += list(zip(*[t[i:] for i in range(n)]))   # общий вид для любого n
    return result
```

Запись `zip(*[t[i:] for i in range(n)])` — то же, что `zip(t, t[1:], ..., t[n-1:])`.

## Стоп-слова в n-граммах

Самые частые биграммы почти всегда содержат служебные слова: *(tili, uchun)*, *(qilish, va)*. Для поиска терминов и устойчивых сочетаний отбрасывают биграммы, в которых **хотя бы одно** слово — стоп-слово. Для триграмм служебное слово в **середине** допустимо (*talabalar uchun yangi*), а по краям — нет.

```python
content_bigrams = Counter({bg: c for bg, c in bigram_freq.items()
                           if bg[0] not in STOPWORDS and bg[1] not in STOPWORDS})
```

## Сравнение категорий

Чтобы сравнить лексику двух групп текстов (например, рубрик *taʼlim* и *turizm*), постройте частоты на 1000 слов для каждой группы и сведите их в одну таблицу:

| слово | taʼlim | turizm |
|---|---|---|
| talabalar | 43.2 | 0.0 |
| sayyohlar | 0.0 | 17.0 |

Слова с высокими частотами в одной группе и нулевыми в другой — **маркеры темы**. Более строгие методы выделения ключевых слов (отношение частот, TF-IDF, PMI) разберём в Л6.

> **Заметка: Сколько данных нужно**  
> На 16 коротких текстах большинство n-грамм встречается один раз — статистика «шумная». Устойчивые выводы о сочетаемости слов делают на корпусах в миллионы токенов: там частоты биграмм становятся надёжными.

## Задание 1. Counter и most_common

В подготовленном коде загружен корпус `df` и есть функция `tokenize(text)` (нормализация апострофов, нижний регистр, токены `\w+`).

1. `all_tokens` — все токены корпуса (документы по порядку).
2. `freq` — `Counter` по `all_tokens`.
3. `top10` — 10 самых частых слов (`most_common`).
4. `n_va` — частота союза «va»; `n_ozbek` — частота слова «oʻzbek». Чтобы не набирать знак ʻ вручную, ищите `freq[normalize("o'zbek")]`.
5. `share_top10` — какую долю всех токенов покрывают 10 самых частых слов.

<details><summary>💡 Подсказка</summary>

`all_tokens += tokenize(text)`; `freq.most_common(10)` возвращает список пар (слово, частота); сумма частот топ-10: `sum(c for w, c in top10)`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    """Нормализация + нижний регистр + токены \w+."""
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

In [ ]:
all_tokens = []
for text in df["text"]:
    ...

freq = ...
top10 = ...
n_va = ...
n_ozbek = ...
share_top10 = ...

print(top10)
print("va:", n_va, " oʻzbek:", n_ozbek, " o'zbek без нормализации:", freq["o'zbek"])
print(f"10 слов покрывают {share_top10:.1%} корпуса")

## Задание 2. Частотная таблица в pandas

В `content_tokens` — токены корпуса **без стоп-слов**.

1. Постройте `freq_df` — DataFrame со столбцами `word`, `count`, отсортированный по `count` по убыванию (одна строка — одно слово).
2. Добавьте столбцы `rel_freq` = count / (сумма count) и `per_1000` = rel_freq × 1000.
3. Постройте горизонтальную столбчатую диаграмму 15 самых частых слов.

<details><summary>💡 Подсказка</summary>

`pd.DataFrame(Counter(content_tokens).most_common(), columns=["word", "count"])` — сразу отсортировано. Диаграмма: `top15.plot.barh(x="word", y="count")`, затем `plt.gca().invert_yaxis()` и `plt.show()`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    """Нормализация + нижний регистр + токены \w+."""
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

content_tokens = [t for text in df["text"] for t in tokenize(text) if t not in STOPWORDS]

In [ ]:
import matplotlib.pyplot as plt

freq_df = pd.DataFrame(..., columns=["word", "count"])
freq_df["rel_freq"] = ...
freq_df["per_1000"] = ...

top15 = freq_df.head(15)
# горизонтальная диаграмма: самые частые слова сверху
...
freq_df.head(10)

## Задание 3. Биграммы и триграммы через zip и NLTK

В `text` — новость `uz_14.txt` о компьютерной лингвистике.

1. `toks` — токены текста (`tokenize`).
2. `bigrams` и `trigrams` — списки кортежей, построенные через `zip` со сдвигом.
3. `bigrams_nltk` — биграммы, полученные функцией `ngrams` из `nltk.util` (списком). Совпадают ли они с вашими?
4. `top_bigrams` — 5 самых частых биграмм текста (`Counter(...).most_common(5)`).

Первый импорт `nltk` в браузере занимает несколько секунд.

<details><summary>💡 Подсказка</summary>

`list(zip(toks, toks[1:]))` — биграммы, `list(zip(toks, toks[1:], toks[2:]))` — триграммы. В NLTK: `list(ngrams(toks, 2))` — без `list` получится генератор.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    """Нормализация + нижний регистр + токены \w+."""
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

text = open("data/corpus_uz/uz_14.txt", encoding="utf-8").read()

In [ ]:
from nltk.util import ngrams

toks = ...
bigrams = ...
trigrams = ...
bigrams_nltk = ...
top_bigrams = ...

print(len(toks), "токенов,", len(bigrams), "биграмм,", len(trigrams), "триграмм")
print("zip == nltk:", bigrams == bigrams_nltk)
print(top_bigrams)

## Задание 4. N-граммы внутри предложений

Функция `split_sentences(text)` уже определена (деление по `. ! ?` и переводам строк).

1. Напишите `sentence_ngrams(text, n)` — список n-грамм (кортежей), построенных **отдельно в каждом предложении**: для каждого предложения возьмите `tokenize(sent)` и постройте n-граммы через `zip`.
2. `bigram_freq`, `trigram_freq` — `Counter` биграмм и триграмм по всем документам корпуса.
3. `n_cross` — сколько «ложных» биграмм появилось бы, если строить биграммы по всему тексту документа без деления на предложения (сумма по корпусу).

<details><summary>💡 Подсказка</summary>

Внутри цикла: `result += list(zip(*[t[i:] for i in range(n)]))`. Для корпуса: `bigram_freq.update(sentence_ngrams(text, 2))`. Наивных биграмм в документе `len(tokenize(text)) - 1`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    """Нормализация + нижний регистр + токены \w+."""
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

def split_sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+|\n+", text) if s.strip()]

In [ ]:
def sentence_ngrams(text, n):
    result = []
    for sent in split_sentences(text):
        t = tokenize(sent)
        result += ...
    return result

print(sentence_ngrams("Bugun havo issiq. Ertaga yomg'ir yog'adi!", 2))

bigram_freq = Counter()
trigram_freq = Counter()
for text in df["text"]:
    ...

naive_total = ...       # биграммы по всему документу без деления на предложения
n_cross = ...

print("Биграмм внутри предложений:", sum(bigram_freq.values()))
print("Ложных биграмм на стыках:", n_cross)
print(bigram_freq.most_common(10))

## Задание 5. Фильтрация n-грамм со стоп-словами

`bigram_freq` и `trigram_freq` уже посчитаны (внутри предложений), `STOPWORDS` загружены. Посмотрите на `bigram_freq.most_common(10)` — сколько там служебных слов?

1. `content_bigrams` — `Counter` только тех биграмм, где **ни одно** слово не стоп-слово.
2. `content_trigrams` — `Counter` триграмм, у которых **первое и последнее** слова не стоп-слова (середина может быть любой).
3. `collocations` — список строк вида `"sunʼiy intellekt"` для биграмм из `content_bigrams` с частотой **не меньше 3**, по убыванию частоты.

<details><summary>💡 Подсказка</summary>

Условие для биграммы: `bg[0] not in STOPWORDS and bg[1] not in STOPWORDS`; для триграммы проверяйте `tg[0]` и `tg[2]`. Строки: `[" ".join(bg) for bg, c in content_bigrams.most_common() if c >= 3]`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    """Нормализация + нижний регистр + токены \w+."""
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

def split_sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+|\n+", text) if s.strip()]

def sentence_ngrams(text, n):
    result = []
    for sent in split_sentences(text):
        t = tokenize(sent)
        result += list(zip(*[t[i:] for i in range(n)]))
    return result

bigram_freq, trigram_freq = Counter(), Counter()
for _t in df["text"]:
    bigram_freq.update(sentence_ngrams(_t, 2))
    trigram_freq.update(sentence_ngrams(_t, 3))

In [ ]:
print(bigram_freq.most_common(10))

content_bigrams = Counter({bg: c for bg, c in bigram_freq.items() if ...})
content_trigrams = ...
collocations = ...

print(content_bigrams.most_common(10))
print(content_trigrams.most_common(5))
print(collocations)

## Задание 6. Сравнение двух категорий

Сравните лексику рубрик `CAT_A` («taʼlim», образование) и `CAT_B` («turizm»). Названия уже записаны в переменные — обратите внимание, что в `meta.csv` слово *taʼlim* записано с канонической ʼ.

1. `freq_a`, `freq_b` — `Counter` токенов **без стоп-слов** всех текстов каждой категории.
2. `compare` — DataFrame: индекс — слова, которые встречаются **не меньше 2 раз** хотя бы в одной из категорий; столбцы `CAT_A` и `CAT_B` — частота слова на 1000 токенов (без стоп-слов) этой категории; отсутствующие слова — 0.
3. `common` — отсортированный список слов, которые встречаются в **обеих** категориях.
4. Постройте горизонтальную диаграмму `compare` (две полосы для каждого слова).

<details><summary>💡 Подсказка</summary>

Частота на 1000: `freq_a[w] / total_a * 1000` (для отсутствующего слова Counter вернёт 0). Общие слова: `sorted(set(freq_a) & set(freq_b))`. Диаграмма: `compare.plot.barh(figsize=(7, 8))`.

</details>

In [ ]:
# Подготовительный код задания
import re
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    """Нормализация + нижний регистр + токены \w+."""
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

CAT_A = "ta\u02bclim"   # название категории из meta.csv (со знаком ʼ)
CAT_B = "turizm"

In [ ]:
import matplotlib.pyplot as plt

def category_freq(cat):
    toks = []
    for text in df.loc[df["category"] == cat, "text"]:
        toks += ...
    return Counter(toks)

freq_a = category_freq(CAT_A)
freq_b = category_freq(CAT_B)

words = sorted({w for w, c in freq_a.items() if c >= 2} | ...)
total_a, total_b = sum(freq_a.values()), sum(freq_b.values())
compare = pd.DataFrame({
    CAT_A: [... for w in words],
    CAT_B: [... for w in words],
}, index=words)
common = ...

...
print(common)
compare.sort_values(CAT_A, ascending=False)